In [108]:
# Setup experiment in paths_config.py
# Add the path to the configuration file
# Path to paths_config: /scMEDAL_for_scRNAseq/Experiments/HealthyHeart/paths_config.py
# Only use with python 12
import sys
sys.path.append("../../../")
from paths_config import run_names_dict, results_path_dict, compare_models_path

import os
import numpy as np

from scMEDAL.utils.compare_results_utils_python12 import (
    aggregate_paths,
    read_and_aggregate_scores,
    filter_min_max_silhouette_scores,
    process_all_results,
    process_confidence_intervals,
)


"""
This script reorganizes clustering scores into a single table. 
It also adds 95% confidence intervals (CI) to the results.
Environment: run_models_env
"""

# --------------------------------------------------------------------------------------
# 1. Define dataset type and output directory
# --------------------------------------------------------------------------------------
# Set the dataset type: Use 'val' for development or 'test' for final results
dataset_type = 'test'

# Define output directory for results
out_name = os.path.join(compare_models_path, run_names_dict["run_name_all"])

# Ensure the directory exists
if not os.path.exists(out_name):
    os.makedirs(out_name)
print(f"Directory created or already exists: {out_name}")

# --------------------------------------------------------------------------------------
# 2. Get paths for mean and all scores
# --------------------------------------------------------------------------------------
# Aggregate paths for mean scores
df_all_paths = aggregate_paths(results_path_dict, pattern=f'mean_scores_{dataset_type}_samplesize')
print("\n\nMean scores paths:")
print(df_all_paths.head())

# Aggregate paths for all scores
df_all_paths_allscores = aggregate_paths(results_path_dict, pattern=f'all_scores_{dataset_type}_samplesize')
print("\nAll scores paths:")
print(df_all_paths_allscores)

print("\nColumns in all scores paths:")
print(df_all_paths_allscores.columns)


Directory created or already exists: /endosome/archive/bioinformatics/DLLab/src/AixaAndrade/gitfront/scMEDAL_for_scRNAseq/Experiments/ASD/../outputs/ASD_outputs/compare_models/log_transformed_2916hvggenes/DefineGeneralname4yourexpt
scMEDAL-RE /endosome/archive/bioinformatics/DLLab/src/AixaAndrade/gitfront/scMEDAL_for_scRNAseq/Experiments/ASD/../outputs/ASD_outputs/latent_space/log_transformed_2916hvggenes/scMEDAL-RE/scMEDALpaperresults_run_crossval_loss_recon_weight-110.0_loss_latent_cluster_weight-0.1_n_latent_dims-2_layer_units-512-132_kl_weight-0.0_scaling-min_max_batch_size-512_epochs-500_patience-30_sample_size-10000_2024-07-28_23-18
scVI /endosome/archive/bioinformatics/DLLab/src/AixaAndrade/gitfront/scMEDAL_for_scRNAseq/Experiments/ASD/../outputs/ASD_outputs/latent_space/log_transformed_2916hvggenes/scVI/run_crossval_n_latent_dims-2_n_layers-2_n_hidden-128_gene_likelihood-zinb_dispersion-gene_scaling-min_max_batch_size-512_epochs-500_patience-30_compute_latents_callback-False_sa

In [2]:
def read_and_aggregate_scores(df_all_paths):
    df_list = []

    for _, row in df_all_paths.iterrows():
        path = row['path']
        sample_size = row['sample_size']
        model_name = row['model_name']

        # Load DataFrame with MultiIndex columns
        df = pd.read_csv(path, header=[0, 1])
        print("df.columns:",df.columns)
        print(f"\n\n{df}\n\n")

        # Explicitly identify and rename fold column
        fold_col = [col for col in df.columns if col[0] == 'fold']
        if fold_col:
            df[('fold', '')] = df[fold_col[0]].values
            df.drop(columns=fold_col, inplace=True)
        else:
            df[('fold', '')] = -1  # Default if fold not found

        # Explicitly identify and rename dataset_type column
        dtype_col = [col for col in df.columns if col[0] == 'dataset_type']
        if dtype_col:
            df[('dataset_type', '')] = df[dtype_col[0]].values
            df.drop(columns=dtype_col, inplace=True)
        else:
            df[('dataset_type', '')] = model_name

        # Add metadata explicitly
        df[('sample_size', '')] = sample_size
        df[('latent_name', '')] = model_name
        df[('index', '')] = df.index

        df_list.append(df)

    # Combine all DataFrames clearly and explicitly
    return pd.concat(df_list, ignore_index=True)

In [3]:
df_all_paths_allscores

,sample_size,path,model_name
0,10000,/endosome/archive/bioinformatics/DLLab/src/Aix...,scMEDAL-RE
1,10000,/endosome/archive/bioinformatics/DLLab/src/Aix...,scVI


In [9]:
import pandas as pd

In [89]:
def get_named_columns(df: pd.DataFrame) -> pd.MultiIndex:
    """
    Returns a MultiIndex of columns that do not contain 'Unnamed' in any level.
    
    Parameters:
    - df (pd.DataFrame): A pandas DataFrame with MultiIndex columns.

    Returns:
    - pd.MultiIndex: Columns without 'Unnamed' in any level.
    """
    if not isinstance(df.columns, pd.MultiIndex):
        raise ValueError("DataFrame must have MultiIndex columns")

    mask = ~df.columns.to_frame(index=False).apply(lambda col: col.str.contains("Unnamed"), axis=1).any(axis=1)
    return df.columns[mask]


df= pd.read_csv(df_all_paths_allscores['path'][0], header=[0,1],index_col=None)
cols = get_named_columns(df)
df_new = df.loc[:,cols].copy()
sample_size = df_all_paths_allscores.loc[0,"sample_size"]
latent_name = df_all_paths_allscores.loc[0,"model_name"]
index_ = list(df.index)
df_new["index"]=index_
df_new["fold"]=df["fold"].values.T[0]

df_new["sample_size"] = sample_size
df_new["index"] = index_

In [121]:
df_all_paths_allscores['path'][0]

'/endosome/archive/bioinformatics/DLLab/src/AixaAndrade/gitfront/scMEDAL_for_scRNAseq/Experiments/ASD/../outputs/ASD_outputs/latent_space/log_transformed_2916hvggenes/scMEDAL-RE/scMEDALpaperresults_run_crossval_loss_recon_weight-110.0_loss_latent_cluster_weight-0.1_n_latent_dims-2_layer_units-512-132_kl_weight-0.0_scaling-min_max_batch_size-512_epochs-500_patience-30_sample_size-10000_2024-07-28_23-18/all_scores_test_samplesize-10000.csv'

In [122]:
process_single_model_format(file_path=df_all_paths_allscores['path'][0], model_name="scMEDAL-RE")

reading file: /endosome/archive/bioinformatics/DLLab/src/AixaAndrade/gitfront/scMEDAL_for_scRNAseq/Experiments/ASD/../outputs/ASD_outputs/latent_space/log_transformed_2916hvggenes/scMEDAL-RE/scMEDALpaperresults_run_crossval_loss_recon_weight-110.0_loss_latent_cluster_weight-0.1_n_latent_dims-2_layer_units-512-132_kl_weight-0.0_scaling-min_max_batch_size-512_epochs-500_patience-30_sample_size-10000_2024-07-28_23-18/all_scores_test_samplesize-10000.csv


NaN                                                   \
                  ch                                                    
             batch.1     batch.2 celltype celltype.1  celltype.2 fold   
dataset_type                                                            
scMEDAL-RE      1/db  silhouette       ch       1/db  silhouette  NaN   

                                           0.0                        \
                             436.7425352128527                         
             dataset_type              batch.1               batch.2   
dataset_type                                                           
scMEDAL-RE            NaN  0.09587284694086184  -0.17145341634750366   

                                 ...                  3.0       \
                                 ...    69.23142019953092        
                       celltype  ...           celltype.2 fold   
dataset_type                     ...                             
scMEDAL-RE    261.9781297685927  ...  -0.0883776843547821  4.0   

                                                  4.0                      \
                                    10964.60376759362                       
                     dataset_type             batch.1             batch.2   
dataset_type                                                                
scMEDAL-RE    RE_AE_latent_2_test  0.5000648999037585  0.4056376814842224   

                                                                              \
                                                                               
                        celltype            celltype.1            celltype.2   
dataset_type                                                                   
scMEDAL-RE    27.129652842756677  0.014495737118861087  -0.15831725299358368   

                                        
                                        
             fold         dataset_type  
dataset_type                            
scMEDAL-RE    5.0  RE_AE_latent_2_test  

[1 rows x 42 columns]

In [125]:
df

Unnamed: 0_level_0         batch                          celltype  \
  Unnamed: 0_level_1            ch      1/db silhouette           ch   
0                  0    436.742535  0.095873  -0.171453   261.978130   
1                  1    581.441024  0.108810  -0.140995   251.489115   
2                  2     67.917069  0.037736  -0.413081  1297.475035   
3                  3     69.231420  0.034046  -0.223528  1908.955905   
4                  4  10964.603768  0.500065   0.405638    27.129653   

                                     fold         dataset_type  
       1/db silhouette Unnamed: 7_level_1   Unnamed: 8_level_1  
0  0.015703  -0.270995                  1  RE_AE_latent_2_test  
1  0.034927  -0.248158                  2  RE_AE_latent_2_test  
2  0.021829  -0.192891                  3  RE_AE_latent_2_test  
3  0.041987  -0.088378                  4  RE_AE_latent_2_test  
4  0.014496  -0.158317                  5  RE_AE_latent_2_test

In [106]:
def get_named_columns(df: pd.DataFrame) -> pd.MultiIndex:
    """
    Returns a MultiIndex of columns that do not contain 'Unnamed' in any level.
    """
    return df.columns[
        ~df.columns.to_frame(index=False).apply(lambda col: col.str.contains("Unnamed"), axis=1).any(axis=1)
    ]

def read_and_aggregate_scores(df_all_paths_allscores):
    """
    Reads score CSVs, keeps only named columns, and adds 'fold', 'index', 'sample_size', and 'latent_name'.
    """
    df_list = []

    for _, row in df_all_paths_allscores.iterrows():
        path = row['path']
        sample_size = row['sample_size']
        model_name = row['model_name']

        df = pd.read_csv(path, header=[0, 1], index_col=None)
        # print(df)
        cols = get_named_columns(df)
        df_new = df.loc[:, cols].copy()

        df_new["index"] = df.index
        df_new["fold"] = df["fold"].values.T[0]
        df_new["sample_size"] = sample_size
        df_new["latent_name"] = model_name

        df_list.append(df_new)

    return pd.concat(df_list, ignore_index=True)

In [112]:
out = read_and_aggregate_scores(df_all_paths_allscores)

df.columns: MultiIndex([('Unnamed: 0_level_0', 'Unnamed: 0_level_1'),
            (             'batch',                 'ch'),
            (             'batch',               '1/db'),
            (             'batch',         'silhouette'),
            (          'celltype',                 'ch'),
            (          'celltype',               '1/db'),
            (          'celltype',         'silhouette'),
            (              'fold', 'Unnamed: 7_level_1'),
            (      'dataset_type', 'Unnamed: 8_level_1')],
           )


  Unnamed: 0_level_0         batch                          celltype  \
  Unnamed: 0_level_1            ch      1/db silhouette           ch   
0                  0    436.742535  0.095873  -0.171453   261.978130   
1                  1    581.441024  0.108810  -0.140995   251.489115   
2                  2     67.917069  0.037736  -0.413081  1297.475035   
3                  3     69.231420  0.034046  -0.223528  1908.955905   
4                 

In [113]:
out

Unnamed: 0_level_0         batch                          celltype  \
  Unnamed: 0_level_1            ch      1/db silhouette           ch   
0                0.0    436.742535  0.095873  -0.171453   261.978130   
1                1.0    581.441024  0.108810  -0.140995   251.489115   
2                2.0     67.917069  0.037736  -0.413081  1297.475035   
3                3.0     69.231420  0.034046  -0.223528  1908.955905   
4                4.0  10964.603768  0.500065   0.405638    27.129653   
5                NaN     15.789402  0.024478  -0.177803  7961.576645   
6                NaN     11.993127  0.017962  -0.177296  7762.190689   
7                NaN     11.626820  0.022950  -0.179101  8631.403896   
8                NaN     13.528066  0.024128  -0.212296  8579.386810   
9                NaN     14.321249  0.015897  -0.200440  8617.400530   

                       fold         dataset_type sample_size latent_name index  
       1/db silhouette                                                          
0  0.015703  -0.270995    1  RE_AE_latent_2_test       10000  scMEDAL-RE     0  
1  0.034927  -0.248158    2  RE_AE_latent_2_test       10000  scMEDAL-RE     1  
2  0.021829  -0.192891    3  RE_AE_latent_2_test       10000  scMEDAL-RE     2  
3  0.041987  -0.088378    4  RE_AE_latent_2_test       10000  scMEDAL-RE     3  
4  0.014496  -0.158317    5  RE_AE_latent_2_test       10000  scMEDAL-RE     4  
5  0.463184   0.341057    1   scVI_latent_2_test       10000        scVI     0  
6  0.373772   0.306177    2   scVI_latent_2_test       10000        scVI     1  
7  0.330516   0.350104    3   scVI_latent_2_test       10000        scVI     2  
8  0.337105   0.323040    4   scVI_latent_2_test       10000        scVI     3  
9  0.394302   0.326058    5   scVI_latent_2_test       10000        scVI     4

In [134]:
df_all_paths_allscores['path'][0]

'/endosome/archive/bioinformatics/DLLab/src/AixaAndrade/gitfront/scMEDAL_for_scRNAseq/Experiments/ASD/../outputs/ASD_outputs/latent_space/log_transformed_2916hvggenes/scMEDAL-RE/scMEDALpaperresults_run_crossval_loss_recon_weight-110.0_loss_latent_cluster_weight-0.1_n_latent_dims-2_layer_units-512-132_kl_weight-0.0_scaling-min_max_batch_size-512_epochs-500_patience-30_sample_size-10000_2024-07-28_23-18/all_scores_test_samplesize-10000.csv'

In [110]:
df_allscores

Unnamed: 0_level_0         batch                          celltype  \
  Unnamed: 0_level_1            ch      1/db silhouette           ch   
0                0.0    436.742535  0.095873  -0.171453   261.978130   
1                1.0    581.441024  0.108810  -0.140995   251.489115   
2                2.0     67.917069  0.037736  -0.413081  1297.475035   
3                3.0     69.231420  0.034046  -0.223528  1908.955905   
4                4.0  10964.603768  0.500065   0.405638    27.129653   
5                NaN     15.789402  0.024478  -0.177803  7961.576645   
6                NaN     11.993127  0.017962  -0.177296  7762.190689   
7                NaN     11.626820  0.022950  -0.179101  8631.403896   
8                NaN     13.528066  0.024128  -0.212296  8579.386810   
9                NaN     14.321249  0.015897  -0.200440  8617.400530   

                       fold         dataset_type sample_size latent_name index  
       1/db silhouette                                                          
0  0.015703  -0.270995    1  RE_AE_latent_2_test       10000  scMEDAL-RE     0  
1  0.034927  -0.248158    2  RE_AE_latent_2_test       10000  scMEDAL-RE     1  
2  0.021829  -0.192891    3  RE_AE_latent_2_test       10000  scMEDAL-RE     2  
3  0.041987  -0.088378    4  RE_AE_latent_2_test       10000  scMEDAL-RE     3  
4  0.014496  -0.158317    5  RE_AE_latent_2_test       10000  scMEDAL-RE     4  
5  0.463184   0.341057    1   scVI_latent_2_test       10000        scVI     0  
6  0.373772   0.306177    2   scVI_latent_2_test       10000        scVI     1  
7  0.330516   0.350104    3   scVI_latent_2_test       10000        scVI     2  
8  0.337105   0.323040    4   scVI_latent_2_test       10000        scVI     3  
9  0.394302   0.326058    5   scVI_latent_2_test       10000        scVI     4

In [135]:
import scipy.stats as stats
def get_CI(df, dof, mean_col, sem_col):
    # Critical t-value for 95% confidence interval
    t_critical = stats.t.ppf(1 - 0.025, dof)

    # Calculate the margin of error
    df['margin_of_error'] = df[sem_col] * t_critical

        # Calculate the confidence intervals
    df[(mean_col[0], mean_col[1], 'CI_lower')] = df[mean_col] - df['margin_of_error']
    df[(mean_col[0], mean_col[1], 'CI_upper')] = df[mean_col] + df['margin_of_error']
    return df[[(mean_col[0], mean_col[1], 'CI_lower'), (mean_col[0], mean_col[1], 'CI_upper')]]

In [129]:
pd.read_csv(df_all_paths_allscores['path'][0], header=[0,1])

Unnamed: 0_level_0         batch                          celltype  \
  Unnamed: 0_level_1            ch      1/db silhouette           ch   
0                  0    436.742535  0.095873  -0.171453   261.978130   
1                  1    581.441024  0.108810  -0.140995   251.489115   
2                  2     67.917069  0.037736  -0.413081  1297.475035   
3                  3     69.231420  0.034046  -0.223528  1908.955905   
4                  4  10964.603768  0.500065   0.405638    27.129653   

                                     fold         dataset_type  
       1/db silhouette Unnamed: 7_level_1   Unnamed: 8_level_1  
0  0.015703  -0.270995                  1  RE_AE_latent_2_test  
1  0.034927  -0.248158                  2  RE_AE_latent_2_test  
2  0.021829  -0.192891                  3  RE_AE_latent_2_test  
3  0.041987  -0.088378                  4  RE_AE_latent_2_test  
4  0.014496  -0.158317                  5  RE_AE_latent_2_test

In [136]:
df_allscores

Unnamed: 0_level_0         batch                          celltype  \
  Unnamed: 0_level_1            ch      1/db silhouette           ch   
0                0.0    436.742535  0.095873  -0.171453   261.978130   
1                1.0    581.441024  0.108810  -0.140995   251.489115   
2                2.0     67.917069  0.037736  -0.413081  1297.475035   
3                3.0     69.231420  0.034046  -0.223528  1908.955905   
4                4.0  10964.603768  0.500065   0.405638    27.129653   
5                NaN     15.789402  0.024478  -0.177803  7961.576645   
6                NaN     11.993127  0.017962  -0.177296  7762.190689   
7                NaN     11.626820  0.022950  -0.179101  8631.403896   
8                NaN     13.528066  0.024128  -0.212296  8579.386810   
9                NaN     14.321249  0.015897  -0.200440  8617.400530   

                       fold         dataset_type sample_size latent_name index  
       1/db silhouette                                                          
0  0.015703  -0.270995    1  RE_AE_latent_2_test       10000  scMEDAL-RE     0  
1  0.034927  -0.248158    2  RE_AE_latent_2_test       10000  scMEDAL-RE     1  
2  0.021829  -0.192891    3  RE_AE_latent_2_test       10000  scMEDAL-RE     2  
3  0.041987  -0.088378    4  RE_AE_latent_2_test       10000  scMEDAL-RE     3  
4  0.014496  -0.158317    5  RE_AE_latent_2_test       10000  scMEDAL-RE     4  
5  0.463184   0.341057    1   scVI_latent_2_test       10000        scVI     0  
6  0.373772   0.306177    2   scVI_latent_2_test       10000        scVI     1  
7  0.330516   0.350104    3   scVI_latent_2_test       10000        scVI     2  
8  0.337105   0.323040    4   scVI_latent_2_test       10000        scVI     3  
9  0.394302   0.326058    5   scVI_latent_2_test       10000        scVI     4

In [114]:
# Process all results
df_sample_size = process_all_results(df_all_paths, models2process_dict, out_name, dataset_type)



Sample size: 10000
scMEDAL-RE file path: /endosome/archive/bioinformatics/DLLab/src/AixaAndrade/gitfront/scMEDAL_for_scRNAseq/Experiments/ASD/../outputs/ASD_outputs/latent_space/log_transformed_2916hvggenes/scMEDAL-RE/scMEDALpaperresults_run_crossval_loss_recon_weight-110.0_loss_latent_cluster_weight-0.1_n_latent_dims-2_layer_units-512-132_kl_weight-0.0_scaling-min_max_batch_size-512_epochs-500_patience-30_sample_size-10000_2024-07-28_23-18/mean_scores_test_samplesize-10000.csv
reading file: /endosome/archive/bioinformatics/DLLab/src/AixaAndrade/gitfront/scMEDAL_for_scRNAseq/Experiments/ASD/../outputs/ASD_outputs/latent_space/log_transformed_2916hvggenes/scMEDAL-RE/scMEDALpaperresults_run_crossval_loss_recon_weight-110.0_loss_latent_cluster_weight-0.1_n_latent_dims-2_layer_units-512-132_kl_weight-0.0_scaling-min_max_batch_size-512_epochs-500_patience-30_sample_size-10000_2024-07-28_23-18/mean_scores_test_samplesize-10000.csv
scVI file path: /endosome/archive/bioinformatics/DLLab/src/A

In [132]:
df

Unnamed: 0_level_0         batch                          celltype  \
  Unnamed: 0_level_1            ch      1/db silhouette           ch   
0                  0    436.742535  0.095873  -0.171453   261.978130   
1                  1    581.441024  0.108810  -0.140995   251.489115   
2                  2     67.917069  0.037736  -0.413081  1297.475035   
3                  3     69.231420  0.034046  -0.223528  1908.955905   
4                  4  10964.603768  0.500065   0.405638    27.129653   

                                     fold         dataset_type  
       1/db silhouette Unnamed: 7_level_1   Unnamed: 8_level_1  
0  0.015703  -0.270995                  1  RE_AE_latent_2_test  
1  0.034927  -0.248158                  2  RE_AE_latent_2_test  
2  0.021829  -0.192891                  3  RE_AE_latent_2_test  
3  0.041987  -0.088378                  4  RE_AE_latent_2_test  
4  0.014496  -0.158317                  5  RE_AE_latent_2_test

In [133]:
df_new

batch                          celltype                      index  \
             ch      1/db silhouette           ch      1/db silhouette         
0    436.742535  0.095873  -0.171453   261.978130  0.015703  -0.270995     0   
1    581.441024  0.108810  -0.140995   251.489115  0.034927  -0.248158     1   
2     67.917069  0.037736  -0.413081  1297.475035  0.021829  -0.192891     2   
3     69.231420  0.034046  -0.223528  1908.955905  0.041987  -0.088378     3   
4  10964.603768  0.500065   0.405638    27.129653  0.014496  -0.158317     4   

  fold sample_size  
                    
0    1       10000  
1    2       10000  
2    3       10000  
3    4       10000  
4    5       10000

In [130]:
def process_single_model_format(file_path, model_name):
    """
    Process a CSV with (index=[0,1]) and multi-index columns, dropping 'fold' if present at any level.
    """
    print("reading file:", file_path)

    # Load DataFrame (MultiIndex on index, single-level columns)
    # df = pd.read_csv(file_path, header=[0], index_col=[0, 1]).T
    df = pd.read_csv(file_path, header=[0,1])

    # Drop any columns with 'fold' in any level of the column tuple
    df = df[[col for col in df.columns if 'fold' not in col]]

    # Convert to dict for flattening
    data = df.to_dict()

    # Build MultiIndex columns
    columns = pd.MultiIndex.from_tuples([(key[0], key[1], stat) for key in data for stat in data[key]])
    values = [data[key][stat] for key in data for stat in data[key]]

    # Recreate DataFrame
    new_df = pd.DataFrame([values], columns=columns)
    new_df["dataset_type"] = model_name
    new_df.set_index("dataset_type", inplace=True)

    return new_df

In [131]:
process_single_model_format(file_path= df_all_paths_allscores['path'][0], model_name="scMEDAL-RE")

reading file: /endosome/archive/bioinformatics/DLLab/src/AixaAndrade/gitfront/scMEDAL_for_scRNAseq/Experiments/ASD/../outputs/ASD_outputs/latent_space/log_transformed_2916hvggenes/scMEDAL-RE/scMEDALpaperresults_run_crossval_loss_recon_weight-110.0_loss_latent_cluster_weight-0.1_n_latent_dims-2_layer_units-512-132_kl_weight-0.0_scaling-min_max_batch_size-512_epochs-500_patience-30_sample_size-10000_2024-07-28_23-18/all_scores_test_samplesize-10000.csv


Unnamed: 0_level_0        \
                                                   Unnamed: 0_level_1         
                                                                    0  1  2   
dataset_type                                                                  
(scMEDAL-RE, scMEDAL-RE, scMEDAL-RE, scMEDAL-RE...                  0  1  2   

                                                               batch  \
                                                                  ch   
                                                    3  4           0   
dataset_type                                                           
(scMEDAL-RE, scMEDAL-RE, scMEDAL-RE, scMEDAL-RE...  3  4  436.742535   

                                                                           \
                                                                            
                                                             1          2   
dataset_type                                                                
(scMEDAL-RE, scMEDAL-RE, scMEDAL-RE, scMEDAL-RE...  581.441024  67.917069   

                                                                            \
                                                                             
                                                           3             4   
dataset_type                                                                 
(scMEDAL-RE, scMEDAL-RE, scMEDAL-RE, scMEDAL-RE...  69.23142  10964.603768   

                                                    ...  celltype            \
                                                    ...      1/db             
                                                    ...         0         1   
dataset_type                                        ...                       
(scMEDAL-RE, scMEDAL-RE, scMEDAL-RE, scMEDAL-RE...  ...  0.015703  0.034927   

                                                                        \
                                                                         
                                                           2         3   
dataset_type                                                             
(scMEDAL-RE, scMEDAL-RE, scMEDAL-RE, scMEDAL-RE...  0.021829  0.041987   

                                                                         \
                                                             silhouette   
                                                           4          0   
dataset_type                                                              
(scMEDAL-RE, scMEDAL-RE, scMEDAL-RE, scMEDAL-RE...  0.014496  -0.270995   

                                                                        \
                                                                         
                                                           1         2   
dataset_type                                                             
(scMEDAL-RE, scMEDAL-RE, scMEDAL-RE, scMEDAL-RE... -0.248158 -0.192891   

                                                                        
                                                                        
                                                           3         4  
dataset_type                                                            
(scMEDAL-RE, scMEDAL-RE, scMEDAL-RE, scMEDAL-RE... -0.088378 -0.158317  

[1 rows x 35 columns]

In [117]:
def process_all_results(df_all_paths, models2process_dict, out_name, dataset_type):
    """
    Process and merge results from different models and sample sizes into a single DataFrame, and save to CSV.

    Parameters:
    df_all_paths (pd.DataFrame): DataFrame with columns 'sample_size', 'model_name', and 'path'.
    models2process_dict (dict): Dict indicating the processing type for each model.
    out_name (str): Directory to save processed CSVs.
    dataset_type (str): Label for the dataset type being processed.

    Returns:
    dict: sample_size ? DataFrame mapping.
    """
    df_sample_size = {}

    for sample_size in np.unique(df_all_paths["sample_size"]):
        print(f"\nSample size: {sample_size}")
        df_smf_list = []
        df_pcaf = pd.DataFrame()
        count_i = 0

        for model_name in np.unique(df_all_paths["model_name"]):
            file_paths = df_all_paths.loc[
                (df_all_paths["sample_size"] == sample_size) & 
                (df_all_paths["model_name"] == model_name), 
                "path"
            ].values

            if len(file_paths) == 0:
                print(f"{model_name} has no path for sample size {sample_size}")
                continue

            file_path = file_paths[0]
            print(f"{model_name} file path: {file_path}")

            process_type = models2process_dict.get(model_name, None)

            if process_type == "process_single_model_format":
                df_smf_list.append(process_single_model_format(file_path=file_path, model_name=model_name))
            
            elif process_type == "preprocess_results_model_pca_format":
                df_pcaf_i = preprocess_results_model_pca_format(file_path, columns_to_drop=['fold', 'sem_fold'])
                df_pcaf_i = df_pcaf_i.rename(columns={"X_pca_val": f"X_pca_val_{count_i}"})
                df_pcaf = pd.concat([df_pcaf, df_pcaf_i], axis=1)
                count_i += 1

            else:
                print(f"{model_name} not processed (unknown format)")

        df_smf = pd.concat(df_smf_list).T if df_smf_list else pd.DataFrame()

        # Combine both formats
        df_all = pd.concat([df_pcaf, df_smf], axis=1).T
        output_path = os.path.join(out_name, f"{dataset_type}_scores_{sample_size}.csv")
        df_all.to_csv(output_path)
        print(f"Saved: {output_path}")
        df_sample_size[sample_size] = df_all

    return df_sample_size

In [ ]:
def process_single_model_format(file_path, model_name):
    """
    Process a CSV with (index=[0,1]) and multi-index columns, dropping 'fold' if present at any level.
    """
    print("reading file:", file_path)

    # Load DataFrame (MultiIndex on index, single-level columns)
    df = pd.read_csv(file_path, header=[0], index_col=[0, 1]).T

    # Drop any columns with 'fold' in any level of the column tuple
    df = df[[col for col in df.columns if 'fold' not in col]]

    # Convert to dict for flattening
    data = df.to_dict()

    # Build MultiIndex columns
    columns = pd.MultiIndex.from_tuples([(key[0], key[1], stat) for key in data for stat in data[key]])
    values = [data[key][stat] for key in data for stat in data[key]]

    # Recreate DataFrame
    new_df = pd.DataFrame([values], columns=columns)
    new_df["dataset_type"] = model_name
    new_df.set_index("dataset_type", inplace=True)

    return new_df

In [116]:
df_sample_size["10000"]

batch                                                \
                       ch                                1/db             
                     mean          std          sem      mean       std   
dataset_type                                                              
scMEDAL-RE    2423.987163  4779.701622  2137.547548  0.155306  0.195627   
scVI                  NaN          NaN          NaN       NaN       NaN   

                                                         celltype  ...  \
                       silhouette                              ch  ...   
                   sem       mean       std       sem        mean  ...   
dataset_type                                                       ...   
scMEDAL-RE    0.087487  -0.108684  0.306332  0.136996  749.405568  ...   
scVI               NaN        NaN       NaN       NaN         NaN  ...   

                  NaN                               scVI_latent_2_test  \
                   ch                               13.451732782120732   
             celltype celltype.1  celltype.2 fold              batch.1   
dataset_type                                                             
scMEDAL-RE        NaN        NaN         NaN  NaN                  NaN   
scVI               ch       1/db  silhouette  NaN  0.02108283988092098   

                                                                            \
                                                                             
                           batch.2           celltype           celltype.1   
dataset_type                                                                 
scMEDAL-RE                     NaN                NaN                  NaN   
scVI          -0.18938742578029633  8310.391713847697  0.37977571305703806   

                                        
                                        
                       celltype.2 fold  
dataset_type                            
scMEDAL-RE                    NaN  NaN  
scVI          0.32928714752197263  3.0  

[2 rows x 30 columns]

In [109]:

# --------------------------------------------------------------------------------------
# 3. Read all scores and save to a CSV file
# --------------------------------------------------------------------------------------
df_allscores = read_and_aggregate_scores(df_all_paths_allscores)
print("\nAggregated scores DataFrame:")
print(df_allscores)

# Save all scores to CSV
df_allscores.to_csv(os.path.join(out_name, f"{dataset_type}_allscores.csv"))

# --------------------------------------------------------------------------------------
# 4. Filter minimum and maximum silhouette scores for the batch
# --------------------------------------------------------------------------------------
for sample_size in np.unique(df_allscores["sample_size"]):
    # Filter minimum and maximum silhouette scores for the "batch" column
    df_min_silhouette, df_max_silhouette = filter_min_max_silhouette_scores(df_allscores, batch_col="batch")
    
    # Save the filtered results to CSV
    df_min_silhouette.to_csv(os.path.join(out_name, f"{dataset_type}_scores_{sample_size}_min_silhouette_batch.csv"))
    df_max_silhouette.to_csv(os.path.join(out_name, f"{dataset_type}_scores_{sample_size}_max_silhouette_batch.csv"))

    # Print the resulting DataFrames
    print("DataFrame with minimum silhouette scores:")
    print(df_min_silhouette)

    print("\nDataFrame with maximum silhouette scores:")
    print(df_max_silhouette)

# --------------------------------------------------------------------------------------
# 5. Process results depending on the model configuration
# --------------------------------------------------------------------------------------
# Define how to process results for each model
# If `get_pca=True` for any model in the pipeline, use "preprocess_results_model_pca_format".
# Otherwise, use "process_single_model_format" or leave empty if no processing is required.
# NOTE: Keys in models2process_dict must match those in run_names_dict.
models2process_dict = {
    # "AE": "preprocess_results_model_pca_format",
    # "AEC": "preprocess_results_model_pca_format",
    # "scMEDAL-FEC": "process_single_model_format",
    # "scMEDAL-FE": "process_single_model_format",
    "scMEDAL-RE": "process_single_model_format",
    "scVI": "process_single_model_format"
}

# Process all results
df_sample_size = process_all_results(df_all_paths, models2process_dict, out_name, dataset_type)

# --------------------------------------------------------------------------------------
# 6. Calculate 95% confidence intervals (CI) for results
# --------------------------------------------------------------------------------------
for sample_size, df_all in df_sample_size.items():
    df_mean_ci_results = process_confidence_intervals(df_all, out_name, dataset_type, sample_size)
    print(f"Sample size: {sample_size}\nConfidence interval results:")
    print(df_mean_ci_results)

df.columns: MultiIndex([('Unnamed: 0_level_0', 'Unnamed: 0_level_1'),
            (             'batch',                 'ch'),
            (             'batch',               '1/db'),
            (             'batch',         'silhouette'),
            (          'celltype',                 'ch'),
            (          'celltype',               '1/db'),
            (          'celltype',         'silhouette'),
            (              'fold', 'Unnamed: 7_level_1'),
            (      'dataset_type', 'Unnamed: 8_level_1')],
           )


  Unnamed: 0_level_0         batch                          celltype  \
  Unnamed: 0_level_1            ch      1/db silhouette           ch   
0                  0    436.742535  0.095873  -0.171453   261.978130   
1                  1    581.441024  0.108810  -0.140995   251.489115   
2                  2     67.917069  0.037736  -0.413081  1297.475035   
3                  3     69.231420  0.034046  -0.223528  1908.955905   
4                 

In [90]:
df_new

batch                          celltype                      index  \
             ch      1/db silhouette           ch      1/db silhouette         
0    436.742535  0.095873  -0.171453   261.978130  0.015703  -0.270995     0   
1    581.441024  0.108810  -0.140995   251.489115  0.034927  -0.248158     1   
2     67.917069  0.037736  -0.413081  1297.475035  0.021829  -0.192891     2   
3     69.231420  0.034046  -0.223528  1908.955905  0.041987  -0.088378     3   
4  10964.603768  0.500065   0.405638    27.129653  0.014496  -0.158317     4   

  fold sample_size  
                    
0    1       10000  
1    2       10000  
2    3       10000  
3    4       10000  
4    5       10000

In [139]:
# Sample input DataFrame
data = {
    ("batch", "ch"): [436.742535, 581.441024, 67.917069, 69.231420, 10964.603768, 15.789402, 11.993127, 11.626820, 13.528066, 14.321249],
    ("batch", "1/db"): [0.095873, 0.108810, 0.037736, 0.034046, 0.500065, 0.024478, 0.017962, 0.022950, 0.024128, 0.015897],
    ("batch", "silhouette"): [-0.171453, -0.140995, -0.413081, -0.223528, 0.405638, -0.177803, -0.177296, -0.179101, -0.212296, -0.200440],
    ("celltype", "ch"): [261.978130, 251.489115, 1297.475035, 1908.955905, 27.129653, 7961.576645, 7762.190689, 8631.403896, 8579.386810, 8617.400530],
    ("celltype", "1/db"): [0.015703, 0.034927, 0.021829, 0.041987, 0.014496, 0.463184, 0.373772, 0.330516, 0.337105, 0.394302],
    ("celltype", "silhouette"): [-0.270995, -0.248158, -0.192891, -0.088378, -0.158317, 0.341057, 0.306177, 0.350104, 0.323040, 0.326058],
    "index": list(range(10)),
    "latent_name": ["RE_AE_latent_2_test"]*5 + ["scVI_latent_2_test"]*5,
    "sample_size": [10000]*10,
    "dataset_type": ["scMEDAL-RE"]*5 + ["scVI"]*5,
}
df_all = pd.DataFrame(data)

# Add a third level for MultiIndex columns
df_all.columns = pd.MultiIndex.from_tuples([(col if isinstance(col, tuple) else ('meta', col) + ('',)) for col in df_all.columns])

def calculate_and_append_ci(df, dof):
    def get_CI(df, dof, mean_col, sem_col):
        t_critical = stats.t.ppf(1 - 0.025, dof)
        df['margin_of_error'] = df[sem_col] * t_critical
        df[(mean_col[0], mean_col[1], 'CI_lower')] = df[mean_col] - df['margin_of_error']
        df[(mean_col[0], mean_col[1], 'CI_upper')] = df[mean_col] + df['margin_of_error']
        return df[[(mean_col[0], mean_col[1], 'CI_lower'), (mean_col[0], mean_col[1], 'CI_upper')]]

    ci_columns = []
    for level_0 in df.columns.levels[0]:
        for level_1 in df.columns.levels[1]:
            mean_col = (level_0, level_1, 'mean')
            sem_col = (level_0, level_1, 'sem')
            if mean_col in df.columns and sem_col in df.columns:
                df_subset = df[[mean_col, sem_col]]
                ci_df = get_CI(df_subset.copy(), dof, mean_col, sem_col)
                ci_columns.append(ci_df)

    ci_df = pd.concat(ci_columns, axis=1) if ci_columns else pd.DataFrame(index=df.index)
    final_df = pd.concat([df, ci_df], axis=1)
    return final_df

def process_confidence_intervals(df_all, out_name, dataset_type, sample_size, dof=4):
    df_ci = calculate_and_append_ci(df_all, dof=dof)
    # mean_ci_columns = [col for col in df_ci.columns if 'mean' in col[2] or 'CI' in col[2]]
    # df_mean_ci = df_ci[mean_ci_columns]

    # intercalated_columns = []
    # for level_0 in df_ci.columns.levels[0]:
    #     for level_1 in df_ci.columns.levels[1]:
    #         mean_col = (level_0, level_1, 'mean')
    #         ci_lower_col = (level_0, level_1, 'CI_lower')
    #         ci_upper_col = (level_0, level_1, 'CI_upper')
    #         if mean_col in df_mean_ci.columns:
    #             intercalated_columns.append(mean_col)
    #         if ci_lower_col in df_mean_ci.columns:
    #             intercalated_columns.append(ci_lower_col)
    #         if ci_upper_col in df_mean_ci.columns:
    #             intercalated_columns.append(ci_upper_col)

    # df_mean_ci = df_mean_ci[intercalated_columns]

    # os.makedirs(out_name, exist_ok=True)
    # output_path = os.path.join(out_name, f"{dataset_type}_scores_{sample_size}_95CI.csv")
    # df_mean_ci.to_csv(output_path)

    #return df_mean_ci
    return df_ci

In [140]:
process_confidence_intervals(df_all, "", "scMEDAL-RE", 10000)

batch                          celltype                       meta  \
             ch      1/db silhouette           ch      1/db silhouette index   
            NaN       NaN        NaN          NaN       NaN        NaN         
0    436.742535  0.095873  -0.171453   261.978130  0.015703  -0.270995     0   
1    581.441024  0.108810  -0.140995   251.489115  0.034927  -0.248158     1   
2     67.917069  0.037736  -0.413081  1297.475035  0.021829  -0.192891     2   
3     69.231420  0.034046  -0.223528  1908.955905  0.041987  -0.088378     3   
4  10964.603768  0.500065   0.405638    27.129653  0.014496  -0.158317     4   
5     15.789402  0.024478  -0.177803  7961.576645  0.463184   0.341057     5   
6     11.993127  0.017962  -0.177296  7762.190689  0.373772   0.306177     6   
7     11.626820  0.022950  -0.179101  8631.403896  0.330516   0.350104     7   
8     13.528066  0.024128  -0.212296  8579.386810  0.337105   0.323040     8   
9     14.321249  0.015897  -0.200440  8617.400530  0.394302   0.326058     9   

                                                 
           latent_name sample_size dataset_type  
                                                 
0  RE_AE_latent_2_test       10000   scMEDAL-RE  
1  RE_AE_latent_2_test       10000   scMEDAL-RE  
2  RE_AE_latent_2_test       10000   scMEDAL-RE  
3  RE_AE_latent_2_test       10000   scMEDAL-RE  
4  RE_AE_latent_2_test       10000   scMEDAL-RE  
5   scVI_latent_2_test       10000         scVI  
6   scVI_latent_2_test       10000         scVI  
7   scVI_latent_2_test       10000         scVI  
8   scVI_latent_2_test       10000         scVI  
9   scVI_latent_2_test       10000         scVI

In [143]:
df_allscores.columns

MultiIndex([('Unnamed: 0_level_0', 'Unnamed: 0_level_1'),
            (             'batch',                 'ch'),
            (             'batch',               '1/db'),
            (             'batch',         'silhouette'),
            (          'celltype',                 'ch'),
            (          'celltype',               '1/db'),
            (          'celltype',         'silhouette'),
            (              'fold',                   ''),
            (      'dataset_type',                   ''),
            (       'sample_size',                   ''),
            (       'latent_name',                   ''),
            (             'index',                   '')],
           )

In [141]:
df_all.columns

MultiIndex([(   'batch',           'ch', nan),
            (   'batch',         '1/db', nan),
            (   'batch',   'silhouette', nan),
            ('celltype',           'ch', nan),
            ('celltype',         '1/db', nan),
            ('celltype',   'silhouette', nan),
            (    'meta',        'index',  ''),
            (    'meta',  'latent_name',  ''),
            (    'meta',  'sample_size',  ''),
            (    'meta', 'dataset_type',  '')],
           )

In [146]:
    def get_CI(df, dof, mean_col, sem_col):
        t_critical = stats.t.ppf(1 - 0.025, dof)
        df['margin_of_error'] = df[sem_col] * t_critical
        df[(mean_col[0], mean_col[1], 'CI_lower')] = df[mean_col] - df['margin_of_error']
        df[(mean_col[0], mean_col[1], 'CI_upper')] = df[mean_col] + df['margin_of_error']
        return df[[(mean_col[0], mean_col[1], 'CI_lower'), (mean_col[0], mean_col[1], 'CI_upper')]]

In [145]:
get_CI(df_allscores, dof=4, mean_col=('celltype','silhouette'), sem_col=('celltype','silhouette'))

ValueError: Item must have length equal to number of levels.

In [147]:
df[('celltype','silhouette')]

0   -0.270995
1   -0.248158
2   -0.192891
3   -0.088378
4   -0.158317
Name: (celltype, silhouette), dtype: float64